
# PyTorch Datasets (the `torch.utils.data` way)

In PyTorch, a **dataset** is the object that knows *how to fetch one training example*, and a **dataloader** is the object that knows *how to batch, shuffle, and load examples efficiently*.

---

## 1) What is a Dataset?

A (map-style) dataset represents a collection of examples:
$$
\mathcal{D} = \{(x_i, y_i)\}_{i=1}^{N}
$$

A **map-style** `Dataset` behaves like an indexable map:
$$
i \mapsto (x_i, y_i)
$$
So it must implement:

- `__len__()` → returns $N$
- `__getitem__(i)` → returns the $i$-th sample $(x_i, y_i)$

### Minimal skeleton
```python
from torch.utils.data import Dataset

class MyDataset(Dataset):
    def __len__(self):
        return N

    def __getitem__(self, idx):
        # return a single sample
        return x_idx, y_idx
````

---

## 2) DataLoader: batching + shuffling + parallel loading

If a dataset returns single samples, the `DataLoader` builds batches:
$$
B = {(x_{i_j}, y_{i_j})}_{j=1}^{m}
$$
where $m$ is the batch size.

```python
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,      # randomize index order each epoch
    num_workers=2,     # use subprocesses to load data
)

for x, y in loader:
    # x: [32, ...], y: [32, ...] depending on your dataset
    pass
```

**Common knobs**

* `batch_size`: number of samples per batch ($m$)
* `shuffle`: typically `True` for training, `False` for validation/test
* `num_workers`: parallel CPU loading (start with 0, then increase)
* `drop_last`: drop the last incomplete batch if dataset size not divisible by batch size

---

## 3) Using built-in datasets (example: torchvision)

Many datasets are ready-to-use (MNIST, CIFAR10, ImageNet wrappers, etc.).

```python
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader

tfm = transforms.Compose([
    transforms.ToTensor(),
    # normalize each channel: x' = (x - μ) / σ
    transforms.Normalize(mean=(0.5,), std=(0.5,))
])

train_ds = torchvision.datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=tfm
)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)
```

Normalization formula:
$$
x' = \frac{x - \mu}{\sigma}
$$

---

## 4) Writing a Custom Dataset (CSV/tabular example)

Let’s say you have a CSV with features and a label column.

```python
import torch
from torch.utils.data import Dataset
import pandas as pd

class CSVDataset(Dataset):
    def __init__(self, csv_path, feature_cols, label_col, dtype=torch.float32):
        self.df = pd.read_csv(csv_path)
        self.feature_cols = feature_cols
        self.label_col = label_col
        self.dtype = dtype

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = torch.tensor(row[self.feature_cols].values, dtype=self.dtype)
        y = torch.tensor(row[self.label_col], dtype=self.dtype)
        return x, y
```

Usage:

```python
ds = CSVDataset("train.csv", feature_cols=["f1","f2","f3"], label_col="y")
loader = DataLoader(ds, batch_size=32, shuffle=True)
```

---

## 5) Transforms: clean separation of “how to read” vs “how to augment”

A nice pattern: dataset reads raw data, transform handles preprocessing/augmentation.

```python
class WithTransform(Dataset):
    def __init__(self, base_ds, transform=None):
        self.base_ds = base_ds
        self.transform = transform

    def __len__(self):
        return len(self.base_ds)

    def __getitem__(self, idx):
        x, y = self.base_ds[idx]
        if self.transform:
            x = self.transform(x)
        return x, y
```

---



## 6) IterableDataset (streaming / unknown length)

If data is streaming (logs, very large files, online generation), use `IterableDataset`:

* no `__getitem__` / indexing
* you define `__iter__`

```python
from torch.utils.data import IterableDataset

class StreamDataset(IterableDataset):
    def __iter__(self):
        for i in range(10_000):
            x = torch.tensor([i], dtype=torch.float32)
            y = 2 * x + 1
            yield x, y
```

```python
loader = DataLoader(StreamDataset(), batch_size=64)
```

---


# Stock Dataset for CNN

In [8]:
import numpy as np
import pandas as pd
import yfinance as yf
import torch
from torch.utils.data import Dataset, DataLoader

# 1) Download prices -> returns
tickers = ["AAPL","MSFT","AMZN","GOOGL","META","NVDA","TSLA","JPM","XOM","JNJ"]
start = "2018-01-01"

prices = yf.download(tickers, start=start, auto_adjust=True, progress=False)["Close"]

# simple returns: r_t = P_t/P_{t-1} - 1
returns = prices.pct_change().dropna()
returns = returns.dropna(how="any")  # keep only dates where all K exist


class MultiStockWindowCNN2D(Dataset):
    """
    One sample:
      X: [1, w, K]  (channel=1, height=time, width=stocks)
      y: [K, 1] or [K]  (next-day returns for all K stocks)
    """
    def __init__(self, returns_df: pd.DataFrame, window: int, target_2d: bool = True):
        assert window >= 1
        self.window = window
        self.tickers = list(returns_df.columns)
        self.target_2d = target_2d

        R = torch.tensor(returns_df.values, dtype=torch.float32)  # [T, K]
        T, K = R.shape
        self.K = K

        X_seq = R.unfold(dimension=0 , size = window , step = 1)
        X_seq = torch.transpose(X_seq , 1 , 2)
        X_seq = X_seq[:-1]
        y = R[window : ]

        self.X_seq = X_seq
        self.y  = y

        # to do

    def __len__(self):
        return self.X_seq.shape[0]

    def __getitem__(self, idx):
        # to do
        x = self.X_seq[idx].unsqueeze(0)
        y = self.y[idx]
        if self.target_2d :
          y = y.unsqueeze(-1)
        return x, y


# 2) Use it
w = 20
ds = MultiStockWindowCNN2D(returns, window=w, target_2d=True)
loader = DataLoader(ds, batch_size=64, shuffle=True)

x, y = next(iter(loader))
print("x shape:", x.shape)  # [B, 1, w, K]
print("y shape:", y.shape)  # [B, K, 1]
print("K:", ds.K, ds.tickers)

torch.Size([2044, 10])
x shape: torch.Size([64, 1, 20, 10])
y shape: torch.Size([64, 10, 1])
K: 10 ['AAPL', 'AMZN', 'GOOGL', 'JNJ', 'JPM', 'META', 'MSFT', 'NVDA', 'TSLA', 'XOM']


/tmp/ipython-input-84456402.py:14: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change().dropna()


# Introduction: Limit Order Books (LOB) and the FI-2010 Benchmark Dataset

## 1) What is a Limit Order Book (LOB)?

A **Limit Order Book (LOB)** is a real-time record of buy and sell interest for a financial instrument (stock, futures, etc.).  
It contains **limit orders**: orders to buy or sell at a specified price.

At any moment, the LOB is organized into two sides:

- **Bid side** (buy orders): highest buy price is the **best bid**
- **Ask side** (sell orders): lowest sell price is the **best ask**

The difference between them is the **spread**:
$$
\text{Spread}_t = a_t^{(1)} - b_t^{(1)}
$$
where:
- $a_t^{(1)}$ = best ask price at time/event $t$
- $b_t^{(1)}$ = best bid price at time/event $t$

The **mid-price** is the average of best bid and best ask:
$$
m_t = \frac{a_t^{(1)} + b_t^{(1)}}{2}
$$

LOB data is typically recorded at multiple **levels** (depth).  
For level $i = 1,\dots,10$:
- Ask price $a_t^{(i)}$ and ask volume $v_{a,t}^{(i)}$
- Bid price $b_t^{(i)}$ and bid volume $v_{b,t}^{(i)}$

---

## 2) What is the FI-2010 Benchmark Dataset?

The FI-2010 dataset (often used with DeepLOB) is a benchmark for **mid-price movement prediction** using LOB data.

It is an **event-based** dataset:
- Each **row/column index corresponds to an event** (a new order, cancel, trade, etc.)
- For each event, we have the current LOB state (top 10 levels) plus labels for future movement

A common storage format is a matrix with:
- **40 feature rows**
- **5 label rows**
- and **N columns** (events)

So the raw matrix shape is often:
$$
(45, N)
$$

---

## 3) Features: what are the 40 inputs?

The **feature vector** at each event $t$ contains the top-10 LOB levels:

For each level $i=1,\dots,10$ we store:
$$
[a_t^{(i)},\ v_{a,t}^{(i)},\ b_t^{(i)},\ v_{b,t}^{(i)}]
$$

That is **4 numbers per level**, and we have 10 levels:

$$
40 = 10 \times 4
$$

So the feature vector is:
$$
x_t \in \mathbb{R}^{40}
$$

A typical column ordering is:
$$
x_t =
[a_t^{(1)}, v_{a,t}^{(1)}, b_t^{(1)}, v_{b,t}^{(1)},\ \dots,\ a_t^{(10)}, v_{a,t}^{(10)}, b_t^{(10)}, v_{b,t}^{(10)}]
$$

---

## 4) Input sample construction: time window of length T

DeepLOB does not predict from a single event.  
Instead it uses a **window of the most recent T events** (often $T=100$).

The input sample ending at event $t$ is:
$$
X_t = [x_{t-T+1},\ x_{t-T+2},\ \dots,\ x_t] \in \mathbb{R}^{T \times 40}
$$

For CNNs we add a channel dimension:
$$
X_t \in \mathbb{R}^{1 \times T \times 40}
$$

In a PyTorch batch, the shape is:
$$
X \in \mathbb{R}^{B \times 1 \times T \times 40}
$$

---

## 5) Targets: what are we predicting?

The dataset is designed for **mid-price movement forecasting**.

First compute mid-price at event $t$:
$$
m_t = \frac{a_t^{(1)} + b_t^{(1)}}{2}
$$

Then define the **future average mid-price** over a horizon $H$ events:
$$
\bar{m}_{t}^{(H)} = \frac{1}{H}\sum_{i=1}^{H} m_{t+i}
$$

The dataset provides **multiple horizons**, usually 5 horizons (e.g., $H \in \{10, 20, 30, 50, 100\}$ events depending on the provided file/version).

---

## 6) How the class label is computed (Up / Stationary / Down)

For a given horizon $H$, compare the future average mid-price to the current mid-price:

$$
\Delta_t^{(H)} = \bar{m}_t^{(H)} - m_t
$$

Using a threshold $\alpha$ (small positive constant), define a **3-class label**:

$$
y_t^{(H)} =
\begin{cases}
2 & \text{if } \Delta_t^{(H)} > \alpha \quad (\text{Up}) \\
1 & \text{if } |\Delta_t^{(H)}| \le \alpha \quad (\text{Stationary}) \\
0 & \text{if } \Delta_t^{(H)} < -\alpha \quad (\text{Down})
\end{cases}
$$

So each event has 5 labels (one per horizon).  
In the dataset files, these labels are often stored as values in $\{1,2,3\}$, and we convert them to $\{0,1,2\}$ for PyTorch:

$$
y \leftarrow y - 1
$$

---

## 7) Summary: what one training example looks like

Given:
- window length $T$ (e.g., 100)
- horizon index $k$ (choose one of the 5 horizons)

Each dataset item returns:

**Features**
- `x`: shape **[1, T, 40]**
- represents the last $T$ LOB events and their 40-dimensional LOB features

**Target**
- `y`: a single integer class in **{0,1,2}**
- represents the mid-price movement class (Down / Stationary / Up) for the chosen horizon

This is the standard setup used to train DeepLOB-style models on FI-2010.

In [1]:
import pandas as pd
import pickle
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report

import torch
import torch.nn.functional as F
from torch.utils import data
import torch.nn as nn
import torch.optim as optim

In [2]:
import os
if not os.path.isfile('data.zip'):
    !wget https://raw.githubusercontent.com/zcakhaa/DeepLOB-Deep-Convolutional-Neural-Networks-for-Limit-Order-Books/master/data/data.zip
    !unzip -n data.zip
    print('data downloaded.')
else:
    print('data already existed.')

data already existed.


In [11]:

from torch.utils.data import Dataset

import torch.nn.functional as F
def prepare_x(data):
    df1 = data[:40, :].T
    return np.array(df1)

def get_label(data):
    lob = data[-5:, :].T
    return lob

def data_classification(X, Y, T):
    [N, D] = X.shape
    df = np.array(X)

    dY = np.array(Y)

    dataY = dY[T - 1:N]

    dataX = np.zeros((N - T + 1, T, D))
    for i in range(T, N + 1):
        dataX[i - T] = df[i - T:i, :]

    return dataX, dataY

def torch_data(x, y):
    x = torch.from_numpy(x)
    x = torch.unsqueeze(x, 1)
    y = torch.from_numpy(y)
    y = F.one_hot(y, num_classes=3)
    return x, y

class Dataset(Dataset):
    """Characterizes a dataset for PyTorch"""
    def __init__(self, data, k, num_classes, T):
        """Initialization"""
        self.k = k
        self.num_classes = num_classes
        self.T = T

        x = prepare_x(data)
        y = get_label(data)
        x, y = data_classification(x, y, self.T)
        y = y[:,self.k] - 1
        self.length = len(x)

        x = torch.from_numpy(x)
        self.x = torch.unsqueeze(x, 1)
        self.y = torch.from_numpy(y)

    def __len__(self):
        """Denotes the total number of samples"""
        return self.length

    def __getitem__(self, index):
        """Generates samples of data"""
        return self.x[index], self.y[index]

In [3]:
import numpy as np


dec_data = np.loadtxt('Train_Dst_NoAuction_DecPre_CF_7.txt')
dec_train = dec_data[:, :int(np.floor(dec_data.shape[1] * 0.8))]
dec_val = dec_data[:, int(np.floor(dec_data.shape[1] * 0.8)):]

dec_test1 = np.loadtxt('Test_Dst_NoAuction_DecPre_CF_7.txt')
dec_test2 = np.loadtxt('Test_Dst_NoAuction_DecPre_CF_8.txt')
dec_test3 = np.loadtxt('Test_Dst_NoAuction_DecPre_CF_9.txt')
dec_test = np.hstack((dec_test1, dec_test2, dec_test3))

print(dec_train.shape, dec_val.shape, dec_test.shape)

(149, 203800) (149, 50950) (149, 139587)


In [ ]:
batch_size = 64

dataset_train = Dataset(data=dec_train, k=4, num_classes=3, T=100)
dataset_val = Dataset(data=dec_val, k=4, num_classes=3, T=100)
dataset_test = Dataset(data=dec_test, k=4, num_classes=3, T=100)

train_loader = torch.utils.data.DataLoader(dataset=dataset_train, batch_size=batch_size, shuffle=True)
val_loader = torch.utils.data.DataLoader(dataset=dataset_val, batch_size=batch_size, shuffle=False)
test_loader = torch.utils.data.DataLoader(dataset=dataset_test, batch_size=batch_size, shuffle=False)

print(dataset_train.x.shape, dataset_train.y.shape)

In [4]:
import numpy as np
import torch
from torch.utils.data import Dataset

class LOBWindowDataset(Dataset):

    def __init__(self, data, k=0, T=100, x_rows=40, y_rows=5, dtype=np.float32):
        assert data.ndim == 2, "data must be 2D"
        assert data.shape[0] >= x_rows + y_rows, "expected at least 45 rows (40 features + 5 labels)"
        self.T = int(T)
        self.k = int(k)


        X_np = data[:x_rows, :].T.astype(dtype, copy=False)


        Y_np = data[-y_rows:, :].T.astype(np.int64, copy=False)

        self.X = torch.from_numpy(X_np)   # [N, 40]
        self.Y = torch.from_numpy(Y_np)   # [N, 5]
        self.N = self.X.shape[0]

        self.length = self.N - self.T + 1
        if self.length <= 0:
            raise ValueError("T is too large for this sequence length")

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        # window: [T, 40]
        x = self.X[idx : idx + self.T]

        # label at end of window: [5] -> pick horizon k
        y = self.Y[idx + self.T - 1, self.k] - 1  # convert 1/2/3 -> 0/1/2

        # add channel dim for CNN: [1, T, 40]
        x = x.unsqueeze(0)

        return x, y.long()

In [5]:
from torch.utils.data import DataLoader

ds = LOBWindowDataset(dec_train, k=0, T=100)   # k=0..4 depending on horizon

dataset_train = LOBWindowDataset(dec_train, k=4, T=100)
dataset_val = LOBWindowDataset(dec_val, k=4, T=100)
train_loader = DataLoader(dataset_train, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(dataset_val, batch_size=64, shuffle=False)


## Model

In [6]:
import torch
import torch.nn as nn

class DeepLOB(nn.Module):
    """
    DeepLOB with channel sizes like the paper:
      - Conv blocks: @16
      - Inception: @32 (3-branch as in your figure)
      - LSTM hidden: 64
    Expects input: [B, 1, 100, 40]
    Returns: logits [B, y_len] (set return_probs=True for softmax probs)
    """
    def __init__(self, y_len=3, return_probs=False):
        super().__init__()
        self.y_len = y_len
        self.return_probs = return_probs
        act = nn.LeakyReLU(negative_slope=0.01, inplace=True)

        # --------------------
        # Convolution blocks (@16)
        # --------------------
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=(1, 2), stride=(1, 2)),   # 40 -> 20
            act,
            nn.BatchNorm2d(16),

            nn.Conv2d(16, 16, kernel_size=(4, 1), padding="same"),
            act,
            nn.BatchNorm2d(16),

            nn.Conv2d(16, 16, kernel_size=(4, 1), padding="same"),
            act,
            nn.BatchNorm2d(16),
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 16, kernel_size=(1, 2), stride=(1, 2)),  # 20 -> 10
            act,
            nn.BatchNorm2d(16),

            nn.Conv2d(16, 16, kernel_size=(4, 1), padding="same"),
            act,
            nn.BatchNorm2d(16),

            nn.Conv2d(16, 16, kernel_size=(4, 1), padding="same"),
            act,
            nn.BatchNorm2d(16),
        )

        self.conv3 = nn.Sequential(
            nn.Conv2d(16, 16, kernel_size=(1, 10)),                # 10 -> 1
            act,
            nn.BatchNorm2d(16),

            nn.Conv2d(16, 16, kernel_size=(4, 1), padding="same"),
            act,
            nn.BatchNorm2d(16),

            nn.Conv2d(16, 16, kernel_size=(4, 1), padding="same"),
            act,
            nn.BatchNorm2d(16),
        )

        # --------------------
        # Inception@32 (3 branches like your figure)
        # Each branch outputs 32 channels -> concat => 96 channels
        # --------------------
        self.inp1 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=(1, 1), padding="same"),
            act,
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=(3, 1), padding="same"),
            act,
            nn.BatchNorm2d(32),
        )

        self.inp2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=(1, 1), padding="same"),
            act,
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=(5, 1), padding="same"),
            act,
            nn.BatchNorm2d(32),
        )

        self.inp3 = nn.Sequential(
            nn.MaxPool2d(kernel_size=(3, 1), stride=(1, 1), padding=(1, 0)),
            nn.Conv2d(16, 32, kernel_size=(1, 1), padding="same"),
            act,
            nn.BatchNorm2d(32),
        )

        # LSTM over time dimension: input_size = 32*3 = 96
        self.lstm = nn.LSTM(input_size=96, hidden_size=64, num_layers=1, batch_first=True)

        # Classifier
        self.fc = nn.Linear(64, self.y_len)

    def forward(self, x):
        # x: [B, 1, 100, 40]
        x = self.conv1(x)   # [B, 16, 100, 20]
        x = self.conv2(x)   # [B, 16, 100, 10]
        x = self.conv3(x)   # [B, 16, 100,  1]

        b1 = self.inp1(x)   # [B, 32, 100, 1]
        b2 = self.inp2(x)   # [B, 32, 100, 1]
        b3 = self.inp3(x)   # [B, 32, 100, 1]
        x = torch.cat((b1, b2, b3), dim=1)  # [B, 96, 100, 1]

        # Prepare for LSTM: [B, time, features]
        x = x.squeeze(-1)          # [B, 96, 100]
        x = x.permute(0, 2, 1)     # [B, 100, 96]

        x, _ = self.lstm(x)        # [B, 100, 64]
        x = x[:, -1, :]            # [B, 64]
        logits = self.fc(x)        # [B, y_len]

        if self.return_probs:
            return torch.softmax(logits, dim=1)
        return logits

In [7]:
!pip install torchinfo

In [8]:
model = DeepLOB()
from torchinfo import summary
summary(model, (1, 1, 100, 40))

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/conv.py:548: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1025.)
  return F.conv2d(


Layer (type:depth-idx)                   Output Shape              Param #
DeepLOB                                  [1, 3]                    --
├─Sequential: 1-1                        [1, 16, 100, 20]          2,176
│    └─Conv2d: 2-1                       [1, 16, 100, 20]          48
├─Sequential: 1-30                       --                        (recursive)
│    └─LeakyReLU: 2-2                    [1, 16, 100, 20]          --
├─Sequential: 1-7                        --                        (recursive)
│    └─BatchNorm2d: 2-3                  [1, 16, 100, 20]          32
│    └─Conv2d: 2-4                       [1, 16, 100, 20]          1,040
├─Sequential: 1-30                       --                        (recursive)
│    └─LeakyReLU: 2-5                    [1, 16, 100, 20]          --
├─Sequential: 1-7                        --                        (recursive)
│    └─BatchNorm2d: 2-6                  [1, 16, 100, 20]          32
│    └─Conv2d: 2-7                       [1

In [ ]:
import torch
import torch.nn as nn

def train_deeplob(
    model,
    train_loader,
    val_loader=None,
    epochs=20,
    lr=1e-3,
    weight_decay=0.0,
    device=None,
):

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)



    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
    }

    for epoch in range(1, epochs + 1):
        # -------- train --------
        model.train()
        train_loss_sum = 0.0
        correct = 0
        total = 0

        for x, y in train_loader:
            x = x.to(device).float()
            y = y.to(device).long()

            optimizer.zero_grad()


            logits = model(x)                 # [B, 3]
            loss = criterion(logits, y)

            loss.backward()



            optimizer.step()

            train_loss_sum += loss.item() * x.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

        train_loss = train_loss_sum / max(total, 1)
        train_acc = correct / max(total, 1)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)

        # -------- validate --------
        if val_loader is not None:
            model.eval()
            val_loss_sum = 0.0
            v_correct = 0
            v_total = 0

            with torch.no_grad():
                for x, y in val_loader:
                    x = x.to(device).float()
                    y = y.to(device).long()

                    logits = model(x)
                    loss = criterion(logits, y)

                    val_loss_sum += loss.item() * x.size(0)
                    preds = logits.argmax(dim=1)
                    v_correct += (preds == y).sum().item()
                    v_total += y.size(0)

            val_loss = val_loss_sum / max(v_total, 1)
            val_acc = v_correct / max(v_total, 1)

            history["val_loss"].append(val_loss)
            history["val_acc"].append(val_acc)

            print(
                f"Epoch {epoch:03d}/{epochs} | "
                f"train loss {train_loss:.4f} acc {train_acc:.4f} | "
                f"val loss {val_loss:.4f} acc {val_acc:.4f}"
            )
        else:
            print(
                f"Epoch {epoch:03d}/{epochs} | "
                f"train loss {train_loss:.4f} acc {train_acc:.4f}"
            )

    return history

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DeepLOB(y_len=3)

history = train_deeplob(
    model,
    train_loader,
    val_loader=val_loader,   # or None
    epochs=50,
    lr=1e-3,
    device=device
)

Epoch 001/50 | train loss 0.7273 acc 0.6513 | val loss 0.8587 acc 0.5969
